# Task 4 — Dataset Preprocessing

## Objective

This notebook creates the shared dataset and image-preprocessing artifacts used by all Task 4 representation-learning experiments. It answers four questions:

1. Is the source metadata complete and suitable for modelling?
2. How should the data be divided into training and final test sets?
3. How should product categories be represented as retrieval labels?
4. Which image-normalization statistics should every model use?

To prevent evaluation leakage, label mappings and image statistics are derived from the training split only.

### Generated artifacts

- `splits/task4/train.csv`
- `splits/task4/test.csv`
- `artifacts/task4/configs/articleType_gender_label_encoder.json`
- `artifacts/task4/configs/image_preprocessing.json`


## 1. Environment and configuration

The following cells locate the project root, import the shared Task 4 configuration, and create the output directories. Centralizing paths and column names in `src.task4.config` ensures that every experiment uses the same splits and preprocessing artifacts.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while (
    PROJECT_ROOT != PROJECT_ROOT.parent
    and not (PROJECT_ROOT / "pyproject.toml").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import json

import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

from src.task4.config import (
    CONFIG_DIR,
    DATA_PATH,
    LABEL_COLUMN,
    LABEL_ID_COLUMN,
    SPLIT_DIR,
)
from src.task4.preprocessing import compute_rgb_mean_std

In [3]:
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

## 2. Inspect the source metadata

Before splitting the dataset, we inspect its size, schema, data types, and missing values. This check determines whether the catalogue requires missing-value treatment or schema correction before training. At minimum, preprocessing requires a valid image `id` and non-missing `articleType` and `gender` values.


In [4]:
df = pd.read_csv(DATA_PATH)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 37745 entries, 0 to 37744
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   id                  37745 non-null  int64
 1   gender              37745 non-null  str  
 2   masterCategory      37745 non-null  str  
 3   subCategory         37745 non-null  str  
 4   articleType         37745 non-null  str  
 5   baseColour          37745 non-null  str  
 6   season              37745 non-null  str  
 7   year                37745 non-null  int64
 8   usage               37745 non-null  str  
 9   productDisplayName  37745 non-null  str  
dtypes: int64(2), str(8)
memory usage: 2.9 MB


### Interpretation

The dataset contains **37,745 product records and 10 metadata columns**. All displayed columns are complete, including the image identifier, product type, and gender fields required by the retrieval task. No missing-value imputation is therefore required for the current metadata.

> `df.info()` does not establish whether IDs are unique or whether every referenced image exists; those conditions should be validated separately if the source dataset changes.


## 3. Create the outer train–test split

We reserve 10% of the catalogue as a final hold-out test set. Iterative multilabel stratification is applied to one-hot representations of `articleType` and `gender` to help preserve both marginal distributions more reliably than a purely random split.

A fixed random seed makes the split reproducible. The test set remains untouched during model selection and hyperparameter tuning.


In [5]:
stratify_cols = [
    "articleType",
    "gender",
]

stratify_features = pd.get_dummies(
    df[stratify_cols].astype(str),
    prefix=stratify_cols,
)

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=42,
)

outer_train_idx, test_idx = next(splitter.split(df, stratify_features))
outer_train_df = df.iloc[outer_train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train:", outer_train_df.shape)
print("Test: ", test_df.shape)

Train: (33968, 10)
Test:  (3777, 10)


### Interpretation

The split assigns **33,968 products to training** and **3,777 products to testing**, matching the intended 90:10 proportion. This provides a large development set while preserving an independent sample for final retrieval evaluation.

The stratification balances `articleType` and `gender` individually, but it does not guarantee that every joint `articleType × gender` combination appears in training. Joint-label coverage is checked next.


## 4. Construct the retrieval labels

Task 4 defines a retrieval class using the joint combination of `articleType` and `gender`. The two fields are cleaned and joined in the form `articleType__gender`, so products with the same article type but different target genders remain distinct classes.

The integer encoder is fitted on the training split only. A category found only in the test set therefore remains unmapped instead of leaking test information into the class vocabulary. The encoded splits and encoder configuration are then saved for downstream notebooks.


In [ ]:
for split_df in (outer_train_df, test_df):
    split_df[LABEL_COLUMN] = (
        split_df["articleType"].str.strip() + "__" + split_df["gender"].str.strip()
    )

classes = sorted(outer_train_df[LABEL_COLUMN].unique().tolist())
label_to_index = {label: index for index, label in enumerate(classes)}
outer_train_df[LABEL_ID_COLUMN] = (
    outer_train_df[LABEL_COLUMN].map(label_to_index).astype("int64")
)
test_df[LABEL_ID_COLUMN] = test_df[LABEL_COLUMN].map(label_to_index).astype("Int64")

unseen_test_labels = sorted(set(test_df[LABEL_COLUMN]) - set(label_to_index))
if unseen_test_labels:
    print(
        f"Warning: {len(unseen_test_labels)} test label(s) are absent from training and have <NA> IDs."
    )
    print(unseen_test_labels)

with (CONFIG_DIR / "articleType_gender_label_encoder.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(
        {
            "label_column": LABEL_COLUMN,
            "label_id_column": LABEL_ID_COLUMN,
            "fit_split": "train",
            "separator": "__",
            "classes": classes,
            "label_to_index": label_to_index,
        },
        file,
        indent=2,
        sort_keys=True,
    )

outer_train_df.to_csv(SPLIT_DIR / "train.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test.csv", index=False)

class_counts = outer_train_df[LABEL_COLUMN].value_counts()
print(f"Saved {len(classes)} classes")
print(f"Classes with fewer than 2 training examples: {(class_counts < 2).sum()}")


['Innerwear Vests__Women', 'Shirts__Girls', 'Tracksuits__Women']
Saved 249 classes
Classes with fewer than 2 training examples: 30


### Interpretation: a long-tailed product catalogue

The training split defines **249 joint retrieval classes**, including **30 classes with fewer than two training examples**. These rare segments reflect a real-world long-tail catalogue and may not provide enough positive examples for class-balanced metric-learning batches.

Three joint labels occur only in the test split and correctly receive missing class IDs:

- `Innerwear Vests__Women`
- `Shirts__Girls`
- `Tracksuits__Women`

These records represent a cold-start scenario because the models never observe the exact category–gender combinations during training. Downstream evaluation should either exclude them from closed-set class-ID metrics or report them separately as unseen-class queries; they should not be silently mixed into metrics requiring valid training-class IDs.


## 5. Define the image-preprocessing contract

Neural networks train more consistently when image channels are normalized using statistics from the training distribution. Each training image is converted to RGB, letterboxed to `128 × 128`, padded with white pixels when necessary, and converted to values in the `[0, 1]` range.

The channel-wise mean and standard deviation are estimated from training images only. The complete specification is saved so that every Task 4 model uses identical input geometry and normalization.


In [ ]:
train_mean, train_std = compute_rgb_mean_std(outer_train_df["id"])
print("Training RGB mean:", np.round(train_mean, 4))
print("Training RGB std: ", np.round(train_std, 4))

image_preprocessing_config = {
    "input_color_mode": "RGB",
    "resize": {
        "method": "letterbox",
        "target_size": [128, 128],
        "interpolation": "bilinear",
        "padding_color_rgb": [255, 255, 255],
        "centering": [0.5, 0.5],
    },
    "tensor": {
        "layout": "CHW",
        "dtype": "float32",
        "value_range_before_normalization": [0.0, 1.0],
    },
    "normalization": {
        "mean_rgb": train_mean,
        "std_rgb": train_std,
    },
}

with (CONFIG_DIR / "image_preprocessing.json").open("w", encoding="utf-8") as file:
    json.dump(image_preprocessing_config, file, indent=2)

### Interpretation and downstream use

The calculated RGB statistics describe the colour distribution seen during training and form the common normalization baseline for all Task 4 experiments. Because they are estimated exclusively from training images, the test distribution remains independent. Persisting the resize and normalization settings also prevents preprocessing differences between models.

At this point, the pipeline has produced reproducible outer splits, a training-fitted retrieval-label vocabulary, and a shared image-transformation configuration. Subsequent notebooks may create inner training–validation splits from `train.csv`, but must leave `test.csv` untouched until final evaluation.
